# Merge the shards and train the heads

**One person runs this, once, after all five shards exist.** It is a CPU
notebook — there is no GPU work here, and asking for one only shortens the
weekly GPU budget the shards need.

What it does:

1. Attaches the five shard Datasets and puts them in **ascending shard order**.
2. Merges them with `scripts/merge_banks.py`.
3. Checks the merged bank against the frozen manifest — the step that catches an
   out-of-order merge, an overlapping shard, or a re-split manifest.
4. Trains the ablation rungs on the cached vectors. Minutes, on CPU.

## Before you start

* **Settings → Accelerator → None.** Stage B trains ~1M-parameter heads on
  cached feature vectors; a GPU buys nothing.
* Attach all five `aigcdet-bank-dinov3l-shard*` Datasets **and** the
  `techjam-aigc-train` Dataset (for the frozen manifest).
* Every shard must be **complete**. A partial shard merges silently and
  contributes a short bank. The Stage A notebook's last cell prints
  `COMPLETE` / `INCOMPLETE`; get that confirmation for all five first.

In [ ]:
BACKBONE = "dinov3l"
SPLITS   = "train,val_internal"     # must match what the shards were extracted with

REPO_URL = "https://github.com/bersamin12/robust-aigc-detection"
BRANCH   = "feat/robust-aigc-detection"
REPO_DIR = "/kaggle/working/robust-aigc-detection"

MANIFEST_GLOB = "/kaggle/input/techjam-aigc-train*/manifest.parquet"
SHARD_GLOB    = "/kaggle/input/aigcdet-bank-dinov3l-shard*"
MERGED_DIR    = f"/kaggle/working/banks/{BACKBONE}"

RUNGS = ["a0", "a1", "a2", "a3"]    # configs/rungs/<name>.yaml

In [ ]:
import glob, importlib, os, subprocess, sys

def sh(argv, **kw):
    print("$", " ".join(str(a) for a in argv))
    return subprocess.run([str(a) for a in argv], check=True, **kw)

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    sh(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", BRANCH])
    sh(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"])
else:
    sh(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR])

if os.path.join(REPO_DIR, "notebooks") not in sys.path:
    sys.path.insert(0, os.path.join(REPO_DIR, "notebooks"))
import kaggle_bootstrap as kb
importlib.reload(kb)

def installed_version(dist):
    try:
        import importlib.metadata as im
        return im.version(dist)
    except Exception:
        return None

plan = kb.install_plan(os.path.join(REPO_DIR, "pyproject.toml"), REPO_DIR,
                       transformers_version=installed_version("transformers"))
for cmd in plan:
    print("   ", " ".join(cmd))
for cmd in plan:
    sh(cmd, capture_output=True, text=True)
print("install done")

## 1. Shard order

`merge_banks` concatenates in the order it is given and re-fingerprints the
result over that concatenation. Sorting the mount paths as *strings* gives
`shard0, shard1, shard10, shard2` — fine for five shards, wrong the moment there
are ten, and wrong silently. `sorted_shard_dirs` orders by the trailing number.

Read the printed order before continuing. It must be 0, 1, 2, 3, 4.

In [ ]:
MANIFEST = sorted(glob.glob(MANIFEST_GLOB))
assert MANIFEST, f"no manifest Dataset attached (looked for {MANIFEST_GLOB})"
MANIFEST = MANIFEST[0]

mounts = glob.glob(SHARD_GLOB)
assert mounts, f"no shard Datasets attached (looked for {SHARD_GLOB})"

# Each Dataset mount may wrap the bank in a directory; find the one holding
# config.json.
def bank_dir(mount):
    hits = sorted(glob.glob(os.path.join(mount, "**", "config.json"),
                            recursive=True))
    assert hits, f"no bank (config.json) under {mount}"
    assert len(hits) == 1, f"more than one bank under {mount}: {hits}"
    return os.path.dirname(hits[0])

SHARDS = kb.sorted_shard_dirs([bank_dir(m) for m in mounts])
for i, d in enumerate(SHARDS):
    st = kb.read_resume_state(d)
    flag = "OK" if st.n_done == st.n_images else "INCOMPLETE"
    print(f"  {i}: {d}\n      {st.n_done}/{st.n_images} images  "
          f"backbone={st.backbone} seed={st.seed}  {flag}")
    assert st.n_done == st.n_images, (
        f"{d} is incomplete ({st.n_done}/{st.n_images}). Merging a partial "
        "shard produces a short bank that nothing downstream will question. "
        "Have its owner resume it first.")
print(f"\nmerging {len(SHARDS)} shards in the order above")

## 2. Merge

`merge_banks` refuses shards that disagree on backbone, dim, view count or seed,
and refuses shards whose `row_id` sets overlap — an overlap means two people ran
the same `SHARD_INDEX`, and the image would be double-counted in every split.

In [ ]:
os.makedirs(os.path.dirname(MERGED_DIR), exist_ok=True)
rc = kb.run_streaming(kb.merge_argv(MERGED_DIR, SHARDS, repo_dir=REPO_DIR))
if rc != 0:
    raise SystemExit("merge failed -- see the playbook in the Stage A notebook")

## 3. Check the merged bank against the frozen manifest

This is the last cheap moment to catch an out-of-order merge or a manifest that
has moved. It compares against the **split-filtered** manifest — the same rows
the shards were extracted from.

> Do **not** pass `--manifest` to `scripts/train_rung.py` instead. That flag
> reads the *whole* manifest, while a training bank covers only
> `train,val_internal`, so the fingerprints cannot match and it rejects a
> perfectly good bank. The check below is the same check, done against the right
> rows.

In [ ]:
# No `root`, and no image Datasets attached: the comparison is over `rel_path`,
# the identity that means the same thing on every machine. The merge session
# never needs the pixels.
print(kb.verify_merged_bank(MERGED_DIR, MANIFEST, splits=SPLITS))

## 4. Stage B — train the rungs

Minutes each, on CPU, on the cached vectors. `train_rung` evaluates on the
bank's own `val_internal` rows, which is why Stage A had to carry both splits.

`val_auc` is the **clean-view** AUC and is not the selection metric;
`val_auc_mean_views` averages over every cached view and is the ladder's actual
thesis. The headline number comes from `scripts/run_ablation.py`, not from here.

In [ ]:
for rung in RUNGS:
    cfg = os.path.join(REPO_DIR, "configs", "rungs", f"{rung}.yaml")
    print(f"\n=== {rung} ===")
    kb.run_streaming([sys.executable,
                      os.path.join(REPO_DIR, "scripts", "train_rung.py"),
                      "--config", cfg, "--bank", MERGED_DIR,
                      "--out", "/kaggle/working/outputs/rungs",
                      "--device", "cpu"])

## What not to do here

* **Do not re-run `scripts/build_dataset.py`.** The manifest is frozen. Every
  bank above is indexed positionally against it; re-splitting silently
  misaligns labels against cached features and nothing errors.
* **Do not re-extract one shard "to be safe" with different parameters.** Shards
  must agree on backbone, seed and view count or the merge refuses them — and if
  they happen to agree while covering different rows, they will not.
* **Do not delete a shard Dataset after merging** until the merged bank has
  passed the check in step 3 and been published itself.